# Ответы на экзаменационные вопросы: NumPy

Ноутбук закрывает вопросы **1, 2, 3, 4, 5, 19, 20** из списка теоретических вопросов, а также
практические задачи из билетов **№24 (задача 2)**, **№25 (задача 2)** и **№5 (задача 2)**.

Датасеты для этого ноутбука не нужны — все массивы генерируются на месте (`np.random.default_rng`).


In [1]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)


## Вопрос 1. Организация массивов в NumPy: хранение данных, создание массивов, принципы реализации операций с едиными исходными данными

### Хранение данных

`numpy.ndarray` — это **однородный** (все элементы одного `dtype`) массив, который хранится как:

- один непрерывный блок памяти (`buffer`) с "сырыми" байтами;
- метаданные: `dtype` (тип и размер элемента в байтах), `shape` (форма), `strides` (шаг в байтах
  для перехода к следующему элементу по каждой оси).

Это принципиально отличается от списка Python, где список — это массив **указателей** на
разнородные объекты `PyObject`. Однородность и непрерывность памяти в NumPy позволяют:

- обращаться к элементу по формуле смещения `offset = Σ index_i * stride_i`, без разыменования
  указателей;
- применять к данным быстрые векторные операции на уровне C/SIMD, а не поэлементный Python-цикл.

### Создание массивов

```python
np.array([1, 2, 3])                 # из существующей последовательности
np.zeros((3, 4)); np.ones((2, 2))    # заполненные константами
np.full((2, 2), 7)
np.empty((2, 2))                     # без инициализации (быстро, но "мусор" в памяти)
np.arange(0, 10, 2)                  # аналог range, но возвращает массив
np.linspace(0, 1, 5)                 # 5 точек, равномерно от 0 до 1 включительно
np.eye(3)                            # единичная матрица
rng.integers(0, 10, size=(3, 3))     # случайные целые
np.fromfunction(lambda i, j: i + j, (3, 3))
```

### Представления (views) и копии — «единые исходные данные»

Многие операции (`reshape`, базовые срезы `a[1:5]`, транспонирование `a.T`) **не копируют** данные,
а создают новый объект `ndarray`, который ссылается на тот же буфер памяти, но с другими
`shape`/`strides`. Это и есть работа "с едиными исходными данными" — несколько представлений одного
и того же блока памяти, что экономит память и время. Изменение элемента через одно представление
видно и через другое.

In [2]:
a = np.arange(12)
b = a.reshape(3, 4)      # view: новые shape/strides, тот же buffer
b[0, 0] = -1
print("a изменился вместе с b?", a[0] == -1, "| a.base is None:", a.base is None, "| b.base is a:", b.base is a)

c = a[a > 0]              # фильтрация по маске -> ВСЕГДА копия (не совпадает по структуре с исходным буфером)
c[0] = -999
print("после изменения c исходный массив a не изменился:", a[1])

print("\nshape:", b.shape, "strides (в байтах):", b.strides, "dtype:", b.dtype, "itemsize:", b.dtype.itemsize)


a изменился вместе с b? True | a.base is None: True | b.base is a: True
после изменения c исходный массив a не изменился: 1

shape: (3, 4) strides (в байтах): (32, 8) dtype: int64 itemsize: 8


## Вопрос 2. Универсальные функции и применение функций по осям в NumPy

**Универсальная функция (ufunc)** — функция NumPy, реализованная в скомпилированном C-коде, которая
применяется **поэлементно** к массивам с поддержкой broadcasting, явной типизации выходов и
дополнительных методов (`reduce`, `accumulate`, `outer`, `at`). Примеры: `np.add`, `np.multiply`,
`np.sin`, `np.exp`, `np.greater`, `np.maximum`.

Ufunc заменяет собой ручной цикл `for` и обеспечивает ускорение в десятки-сотни раз, так как весь
цикл выполняется во внутреннем C-коде без обращений к интерпретатору Python на каждой итерации.

### Параметр `axis` — применение агрегирующих функций по осям

Большинство агрегирующих функций (`sum`, `mean`, `max`, `min`, `std`, `argmax`, ...) принимают
параметр `axis`, задающий, вдоль какой оси "сжимать" массив:

- `axis=0` — операция вдоль строк ⇒ результат "по столбцам" (агрегируем построчно, для каждого столбца);
- `axis=1` — операция вдоль столбцов ⇒ результат "по строкам";
- `axis=None` (по умолчанию) — по всему массиву;
- `keepdims=True` сохраняет "убитую" ось с размером 1, что удобно для дальнейшего broadcasting.

### Методы ufunc: `reduce`, `accumulate`, `outer`

In [3]:
x = rng.integers(0, 10, size=(3, 4))
print("x =\n", x)
print("sum по axis=0 (свернули строки, остались 4 значения - по столбцу):", x.sum(axis=0))
print("sum по axis=1 (свернули столбцы, остались 3 значения - по строке):", x.sum(axis=1))
print("keepdims=True:", x.sum(axis=1, keepdims=True).shape)

# ufunc и его методы: np.add — это ufunc, а sum(x) фактически == np.add.reduce(x)
print("\nnp.add.reduce(x, axis=1) совпадает с x.sum(axis=1):", np.array_equal(np.add.reduce(x, axis=1), x.sum(axis=1)))
print("np.add.accumulate (кумулятивная сумма) по строке 0:", np.add.accumulate(x[0]))
print("np.multiply.outer([1,2,3],[1,2]) — внешнее произведение:\n", np.multiply.outer([1, 2, 3], [1, 2]))


x =
 [[0 7 6 4]
 [4 8 0 6]
 [2 0 5 9]]
sum по axis=0 (свернули строки, остались 4 значения - по столбцу): [ 6 15 11 19]
sum по axis=1 (свернули столбцы, остались 3 значения - по строке): [17 18 16]
keepdims=True: (3, 1)

np.add.reduce(x, axis=1) совпадает с x.sum(axis=1): True
np.add.accumulate (кумулятивная сумма) по строке 0: [ 0  7 13 17]
np.multiply.outer([1,2,3],[1,2]) — внешнее произведение:
 [[1 2]
 [2 4]
 [3 6]]


## Вопрос 3. Принцип распространения значений (broadcasting) при выполнении операций в NumPy

**Broadcasting** — это набор правил, по которым NumPy "растягивает" (концептуально, без реального
копирования данных) массивы меньшей формы, чтобы поэлементная операция стала возможна между
массивами разной формы.

**Алгоритм** (сравнение форм справа налево):

1. Формы дополняются слева единицами до одинаковой длины (числа осей).
2. Оси совместимы, если их размеры равны **или** один из них равен 1.
3. Если хотя бы для одной пары осей размеры не совпадают и ни один не равен 1 — ошибка
   `ValueError: operands could not be broadcast together`.
4. Ось размера 1 "растягивается" (виртуально повторяется) до размера второго операнда.

Итоговая форма результата — поэлементный максимум форм по каждой оси.

In [4]:
# Пример 1: скаляр + матрица (скаляр растягивается по обеим осям)
m = np.ones((3, 3))
print(m + 10)

# Пример 2: матрица (3,4) + вектор-строка (4,) -> вектор растягивается по оси 0
mat = np.zeros((3, 4))
row = np.array([1, 2, 3, 4])
print("\n", mat + row)

# Пример 3: столбец (3,1) + строка (1,4) -> оба растягиваются, результат (3,4)
col = np.array([[0], [10], [20]])
print("\n", col + row)

# Пример несовместимых форм
try:
    np.zeros((3, 4)) + np.zeros((3, 5))
except ValueError as e:
    print("\nОшибка broadcasting:", e)


[[11. 11. 11.]
 [11. 11. 11.]
 [11. 11. 11.]]

 [[1. 2. 3. 4.]
 [1. 2. 3. 4.]
 [1. 2. 3. 4.]]

 [[ 1  2  3  4]
 [11 12 13 14]
 [21 22 23 24]]

Ошибка broadcasting: operands could not be broadcast together with shapes (3,4) (3,5) 


## Вопрос 4. Маскирование и прихотливое индексирование в NumPy

### Булево маскирование (boolean masking)

Индексация массивом той же формы из `True`/`False` — отбирает элементы, где `True`. Результат —
**всегда копия** (в отличие от срезов). Маску удобно строить как результат сравнения/условия
(`x > 5`), а комбинировать маски — через `&`, `|`, `~` (не `and`/`or`/`not`, т.к. они не
векторизованы и требуют одного логического значения).

### Прихотливое (fancy) индексирование

Индексация **массивом целых индексов** (а не срезом) — позволяет выбрать произвольный, в том числе
повторяющийся и переупорядоченный, набор элементов. Также возвращает копию. При индексации сразу по
нескольким осям массивами индексов результат формируется по правилам broadcasting между массивами
индексов.

In [5]:
x = rng.integers(0, 20, size=10)
print("x =", x)

mask = (x > 5) & (x % 2 == 0)          # булева маска: >5 И чётное
print("маска:", mask)
print("x[mask] =", x[mask])

idx = np.array([0, 0, 3, 9])            # fancy indexing: произвольный/повторяющийся порядок
print("\nx[idx] =", x[idx])

grid = rng.integers(0, 10, size=(4, 4))
print("\ngrid =\n", grid)
rows = np.array([0, 1, 2])
cols = np.array([1, 2, 3])
print("grid[rows, cols] (по парам координат, не срез!):", grid[rows, cols])

# комбинация fancy-индексации по строкам и булевой маски по столбцам
print("\nвыбрать строки 0 и 2, а в них — столбцы, где значение в первой строке > 4:")
col_mask = grid[0] > 4
print(grid[np.array([0, 2])][:, col_mask])

# fancy-индексация допускает и запись (в отличие от результата булевой маски "на чтение")
grid2 = grid.copy()
grid2[grid2 > 7] = -1
print("\nзапись по маске (grid2[grid2>7] = -1):\n", grid2)


x = [14 15 14 15 10  2 16  9 10  7]
маска: [ True False  True False  True False  True False  True False]
x[mask] = [14 14 10 16 10]

x[idx] = [14 14 15  7]

grid =
 [[1 9 7 6]
 [4 8 5 4]
 [4 2 0 5]
 [8 0 8 8]]
grid[rows, cols] (по парам координат, не срез!): [9 5 5]

выбрать строки 0 и 2, а в них — столбцы, где значение в первой строке > 4:
[[9 7 6]
 [2 0 5]]

запись по маске (grid2[grid2>7] = -1):
 [[ 1 -1  7  6]
 [ 4 -1  5  4]
 [ 4  2  0  5]
 [-1  0 -1 -1]]


## Вопросы 5, 19, 20. Векторизация в NumPy

### Что такое векторизация и зачем она нужна (вопрос 19, мотивация)

**Векторизация** — способ переписать операцию так, чтобы она выполнялась не через явный цикл
`for` на уровне Python, а через вызов скомпилированных (C) реализаций NumPy сразу над целым
массивом. Цель — избавиться от накладных расходов интерпретатора Python на каждой итерации
(проверка типов, диспетчеризация методов и т.п.), которые в цикле повторяются миллионы раз.
Выигрыш в скорости обычно 10×–100×.

Самый прямой случай — **скалярные функции**: функция, изначально принимающая один скаляр
(`def f(x): return x**2 if x > 0 else 0`), заменяется на выражение, работающее сразу над массивом:
через встроенные ufunc-и, либо, если своя логика не сводится к готовым ufunc, — через
`np.vectorize`.

### `np.vectorize` — ключевые параметры (вопрос 5)

`np.vectorize` оборачивает обычную скалярную Python-функцию так, чтобы её можно было вызывать как
ufunc — с поддержкой broadcasting. **Важно**: это удобство (синтаксический сахар), а не реальное
ускорение — внутри всё равно выполняется Python-цикл, просто скрытый внутри `np.vectorize`.

Ключевые параметры:

- `pyfunc` — исходная скалярная функция;
- `otypes` — строка/список dtype-ов результата (если не указать, NumPy вызовет функцию на первом
  элементе, чтобы определить тип результата — лишний вызов);
- `excluded` — множество имён/индексов аргументов, которые НЕ должны разбиваться поэлементно
  (передаются в функцию как есть, например, константы или флаги);
- `signature` — **обобщённая сигнатура** (generalized universal function signature), которая
  позволяет векторизовать функции, принимающие не скаляр, а целый **вектор** и/или возвращающие
  вектор (вопрос 20, см. ниже).

### Векторизация для векторных функций и обобщённая сигнатура (вопрос 20)

Если функция принимает не одно число, а целый под-массив (например, вектор из `n` компонент, и
возвращает скаляр — как евклидова норма) — обычный `np.vectorize` без `signature` не подходит,
потому что он по умолчанию считает все оси "поэлементными". Для этого случая указывается
**обобщённая сигнатура** вида `'(n)->()'` (вектор длины n на входе → скаляр на выходе) или
`'(n),(n)->()'` (два вектора одинаковой длины → скаляр) и т.д. NumPy тогда трактует последние оси,
упомянутые в сигнатуре, как "внутренние" (обрабатываются функцией целиком), а все остальные,
待ущие оси — как "циклические" (по ним идёт broadcasting, как в обычных ufunc).

Отличие от скалярной векторизации: в сигнатуре явно указано, сколько измерений "съедает" функция
на вход/выход, поэтому такая векторизация подходит для операций типа "норма каждой строки",
"скалярное произведение построчно", "своя агрегирующая функция по последней оси".

In [6]:
# --- Скалярная векторизация (вопросы 5 и 19) ---
def clip_and_square(x, low, high):
    # обычная скалярная функция на чистом Python: работает с одним числом
    if x < low:
        x = low
    elif x > high:
        x = high
    return x ** 2

vec_clip_square = np.vectorize(clip_and_square, otypes=[np.float64], excluded={"low", "high"})

data = rng.uniform(-5, 5, size=8)
print("data:", data)
print("результат (low=-2, high=2 не разбиваются поэлементно, excluded):")
print(vec_clip_square(data, low=-2, high=2))

# --- Векторизация для векторных функций через обобщённую сигнатуру (вопрос 20) ---
def euclidean_norm(v):
    # функция принимает вектор целиком, а не число
    return np.sqrt(np.sum(v ** 2))

vec_norm = np.vectorize(euclidean_norm, signature="(n)->()")

mat = rng.integers(0, 10, size=(5, 3))
print("\nmat =\n", mat)
print("норма каждой строки через np.vectorize(signature='(n)->()'):", vec_norm(mat))
print("сравнение с np.linalg.norm(mat, axis=1):            ", np.linalg.norm(mat, axis=1))

# для сравнения: пример "истинной" векторной операции - готовый ufunc, а не np.vectorize.
# Он быстрее, т.к. полностью реализован в C, без скрытого питоновского цикла:
print("\nвстроенная (настоящая) векторизация скалярной функции x**2 при x>0 иначе 0:")
print(np.where(data > 0, data ** 2, 0.0))


data: [ 1.317  2.581 -1.455  4.707  3.931  2.784 -3.054 -0.333]
результат (low=-2, high=2 не разбиваются поэлементно, excluded):
[1.734 4.    2.116 4.    4.    4.    4.    0.111]

mat =
 [[4 0 5]
 [1 7 6]
 [9 7 3]
 [9 4 3]
 [9 3 0]]
норма каждой строки через np.vectorize(signature='(n)->()'): [ 6.403  9.274 11.79  10.296  9.487]
сравнение с np.linalg.norm(mat, axis=1):             [ 6.403  9.274 11.79  10.296  9.487]

встроенная (настоящая) векторизация скалярной функции x**2 при x>0 иначе 0:
[ 1.734  6.661  0.    22.156 15.454  7.75   0.     0.   ]


## Практика: билет №24, задача 2

> Создать двухмерный массив 30×4, содержащий случайные целые числа от 0 до 100. Интерпретируя
> массив как 30 векторов из 4-х компонент, вернуть массив 5×4, состоящий из векторов с наибольшей
> длиной (евклидовой нормой). Решить средствами NumPy/Pandas, без циклов Python.

In [7]:
arr = rng.integers(0, 101, size=(30, 4))
print("arr (первые 5 строк):\n", arr[:5])

norms = np.linalg.norm(arr, axis=1)              # длина (норма) каждого из 30 векторов, без циклов
top5_idx = np.argsort(norms)[::-1][:5]           # индексы 5 наибольших норм
top5 = arr[top5_idx]

print("\nнормы 30 векторов:\n", norms.round(2))
print("\nиндексы 5 векторов с наибольшей нормой:", top5_idx)
print("\nитоговый массив 5x4:\n", top5)
print("нормы этих векторов:", norms[top5_idx].round(2))
assert top5.shape == (5, 4)


arr (первые 5 строк):
 [[47 80 19 46]
 [13 69 48 33]
 [22 57 67 94]
 [44 16 84 63]
 [70  9 31 77]]

нормы 30 векторов:
 [105.29  91.23 130.61 114.97 108.95 150.64 105.06 126.38 115.59 137.92
 122.43  89.3   53.19 135.83 112.21 102.23 118.47 113.89 113.36 121.07
 129.11  84.79 148.25  98.06  95.36  78.19 131.85 140.33 100.62  65.58]

индексы 5 векторов с наибольшей нормой: [ 5 22 27  9 13]

итоговый массив 5x4:
 [[ 84  43  81  85]
 [ 56  51  79 100]
 [ 16  90  50  94]
 [ 79  78  67  47]
 [ 67  66  47  86]]
нормы этих векторов: [150.64 148.25 140.33 137.92 135.83]


## Практика: билет №25, задача 2

> Построить one-hot encoding для одномерного массива NumPy из целых неотрицательных чисел (длина
> массива и максимальное значение заранее неизвестны). Протестировать на случайно сгенерированном
> массиве. Пример: для `np.array([2, 3, 2, 2, 2, 1])` каждая строка результата — единица в столбце,
> номер которого равен значению элемента, и нули в остальных столбцах; число столбцов равно
> `max(x) + 1`.

In [8]:
def one_hot_encode(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    n_classes = x.max() + 1                       # число столбцов заранее неизвестно -> считаем из данных
    result = np.zeros((x.size, n_classes), dtype=float)
    result[np.arange(x.size), x] = 1.0             # fancy indexing по двум осям сразу, без циклов
    return result


example = np.array([2, 3, 2, 2, 2, 1])
print("пример из условия:\n", one_hot_encode(example))

# тест на случайном массиве со случайной длиной и случайным максимумом
n = int(rng.integers(6, 15))
max_val = int(rng.integers(3, 8))
test_arr = rng.integers(0, max_val + 1, size=n)
encoded = one_hot_encode(test_arr)

print("\nслучайный тестовый массив:", test_arr)
print("shape one-hot:", encoded.shape, "(ожидалось:", (n, max_val + 1), ")")
print(encoded)

# самопроверка: argmax каждой строки должен вернуть исходное значение
assert np.array_equal(encoded.argmax(axis=1), test_arr)
assert np.array_equal(encoded.sum(axis=1), np.ones(n))
print("\nсамопроверка пройдена: argmax(encoded, axis=1) == исходный массив")


пример из условия:
 [[0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]

случайный тестовый массив: [4 4 2 6 0 2 0 2]
shape one-hot: (8, 7) (ожидалось: (8, 7) )
[[0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]]

самопроверка пройдена: argmax(encoded, axis=1) == исходный массив


## Практика: билет №5, задача 2

> Задан двумерный массив `ar1` размерности (25, 4) из случайных целых чисел от 0 до 15. Определить,
> в каких столбцах не менее 5 раз встречается значение, максимальное по своей строке (вывести
> индексы этих столбцов с комментарием). Для столбца, где таких значений больше всего, заменить
> максимумы по строке на -1. Без циклов Python.

In [9]:
ar1 = rng.integers(0, 16, size=(25, 4))
print("ar1 =\n", ar1)

row_max = ar1.max(axis=1, keepdims=True)               # (25, 1) - максимум каждой строки
is_row_max = ar1 == row_max                             # (25, 4) булева маска: True, где стоит максимум строки
                                                          # (если максимум повторяется в строке - True в нескольких столбцах)

counts_per_col = is_row_max.sum(axis=0)                  # сколько раз максимум строки "попал" в каждый столбец
print("\nсколько раз максимум строки встретился в каждом столбце:", counts_per_col)

cols_ge5 = np.where(counts_per_col >= 5)[0]
print(f"столбцы, где максимум строки встречается не менее 5 раз: {cols_ge5}")

best_col = int(np.argmax(counts_per_col))
print(f"столбец с наибольшим количеством таких значений: {best_col} (встретилось {counts_per_col[best_col]} раз)")

ar1_result = ar1.copy()
mask_best_col = is_row_max[:, best_col]                  # строки, где максимум строки лежит именно в best_col
ar1_result[mask_best_col, best_col] = -1

print("\nмассив после замены максимумов строк в столбце", best_col, "на -1:\n", ar1_result)


ar1 =
 [[15  5 14  7]
 [11  7  4 12]
 [15  4 12  4]
 [11 12  7 11]
 [ 4  1  1  7]
 [14  2  7 11]
 [ 3 11  4 12]
 [ 9  8  2  7]
 [13  0 12  7]
 [11 10  6  4]
 [10  2  9  1]
 [10 10  1 12]
 [ 6 12  0  2]
 [ 7  3  5 10]
 [ 2 10  1  2]
 [ 9 12  2  4]
 [14 15  9  7]
 [ 5  9  9  4]
 [ 0  2 15  6]
 [ 7  7 12  1]
 [ 1  4  7 10]
 [ 7  7 15  2]
 [ 9  8  7 12]
 [ 4 15  5  8]
 [ 8 13  7  4]]

сколько раз максимум строки встретился в каждом столбце: [7 8 4 7]
столбцы, где максимум строки встречается не менее 5 раз: [0 1 3]
столбец с наибольшим количеством таких значений: 1 (встретилось 8 раз)

массив после замены максимумов строк в столбце 1 на -1:
 [[15  5 14  7]
 [11  7  4 12]
 [15  4 12  4]
 [11 -1  7 11]
 [ 4  1  1  7]
 [14  2  7 11]
 [ 3 11  4 12]
 [ 9  8  2  7]
 [13  0 12  7]
 [11 10  6  4]
 [10  2  9  1]
 [10 10  1 12]
 [ 6 -1  0  2]
 [ 7  3  5 10]
 [ 2 -1  1  2]
 [ 9 -1  2  4]
 [14 -1  9  7]
 [ 5 -1  9  4]
 [ 0  2 15  6]
 [ 7  7 12  1]
 [ 1  4  7 10]
 [ 7  7 15  2]
 [ 9  8  7 12]
 [ 4 -1  5